In [1]:
#Part 1 - binary classification checks
#scipy and sklearn are only used as references to check code
import sys
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import minimize
from sklearn.linear_model import LogisticRegression

#find the src folder
root = Path.cwd()
while not (root / "src" / "binary.py").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root / "src"))

from data import load_task
from binary import SIGMOID, ATAN, BinaryObjective, softplus, sigmoid, sigma_atan, log_loss, accuracy

rng = np.random.default_rng(0)
LAMBDAS = (0.01, 0.1, 1.0, 10.0)

#load both datasets + show sizes and class counts
data = {name: load_task(name) for name in ("digits01", "breast_cancer")}
pd.DataFrame([d[4] for d in data.values()]).set_index("task")

,N_train,N_test,D,train_counts,test_counts
task,,,,,
digits01,270,90,64,"[133, 137]","[45, 45]"
breast_cancer,426,143,30,"[159, 267]","[53, 90]"


In [2]:
#check 1 - stability
#naive formulas blow up or cancel, ours should stay finite
u = np.array([-1000.0, -50.0, 0.0, 50.0, 1000.0])

with np.errstate(over="ignore"):
    print("naive log(1+e^u):", np.log1p(np.exp(u)))     #inf at the end
print("softplus(u)     :", softplus(u))
print("sigmoid(u)      :", sigmoid(u))

#atan sigma way out in the left tail, true answer is about 1/(pi*|u|)
print(f"atan at u=-1e17: ours={sigma_atan(-1e17):.3e}  naive={0.5 + np.arctan(-1e17) / np.pi:.3e}  true~{1 / (np.pi * 1e17):.3e}")

naive log(1+e^u): [0.00000000e+00 1.92874985e-22 6.93147181e-01 5.00000000e+01
            inf]
softplus(u)     : [0.00000000e+00 1.92874985e-22 6.93147181e-01 5.00000000e+01
 1.00000000e+03]
sigmoid(u)      : [0.00000000e+00 1.92874985e-22 5.00000000e-01 1.00000000e+00
 1.00000000e+00]
atan at u=-1e17: ours=3.183e-18  naive=0.000e+00  true~3.183e-18


In [3]:
#check 2 - gradient and HVP vs finite differences
#errors around 1e-9 / 1e-10 are just finite difference noise, a wrong formula would be ~1
def check(f, w, v, eps = 1e-6):

    g, curv = f.gradient_and_curvature(w)
    g_fd = np.array([ (f.objective(w + eps*e) - f.objective(w - eps*e)) / (2*eps) for e in np.eye(len(w)) ])

    Hv = f.hessian_vector_product(v, curv)
    Hv_fd = (f.gradient_and_curvature(w + eps*v)[0] - f.gradient_and_curvature(w - eps*v)[0]) / (2*eps)

    return np.linalg.norm(g - g_fd) / np.linalg.norm(g), np.linalg.norm(Hv - Hv_fd) / np.linalg.norm(Hv)


rows = []
for name, (Xtr, _, ytr, _, _) in data.items():
    for act in (SIGMOID, ATAN):
        for lam in LAMBDAS:
            f = BinaryObjective(Xtr, ytr, lam, act)
            w = 0.3 * rng.standard_normal(Xtr.shape[1])
            v = rng.standard_normal(Xtr.shape[1])

            gerr, herr = check(f, w, v)
            rows.append(dict(task=name, act=act.name, lam=lam, grad_err=gerr, hvp_err=herr))

pd.DataFrame(rows).pivot_table(index=["task", "act"], columns="lam", values=["grad_err", "hvp_err"])

grad_err                                            \
lam                           0.01          0.10          1.00          10.00   
task          act                                                               
breast_cancer atan     1.447669e-10  9.439045e-11  1.244646e-10  2.228630e-11   
              sigmoid  2.257122e-10  3.921401e-10  2.702113e-10  1.791913e-10   
digits01      atan     8.134363e-11  9.442055e-11  9.322176e-11  9.561977e-11   
              sigmoid  2.305636e-10  2.152207e-10  1.946064e-10  3.339208e-10   

                            hvp_err                                            
lam                           0.01          0.10          1.00          10.00  
task          act                                                              
breast_cancer atan     1.002787e-10  8.273750e-11  1.386867e-10  6.017358e-11  
              sigmoid  5.573515e-11  2.347563e-10  5.854432e-11  3.656847e-11  
digits01      atan     2.338980e-10  1.326004e-10  1.835662e-10  1.551035e-10  
              sigmoid  1.236712e-10  5.763296e-11  6.802321e-11  5.156459e-11

In [4]:
#check 3 - convexity
#build H one column at a time from the HVP this is for checking only
#should be symmetric and every eigenvalue > 0
rows = []
for name, (Xtr, _, ytr, _, _) in data.items():
    for act in (SIGMOID, ATAN):
        f = BinaryObjective(Xtr, ytr, 1.0, act)
        _, curv = f.gradient_and_curvature(0.3 * rng.standard_normal(Xtr.shape[1]))

        H = np.column_stack([ f.hessian_vector_product(e, curv) for e in np.eye(Xtr.shape[1]) ])
        rows.append(dict(task=name, act=act.name, max_asymmetry=np.abs(H - H.T).max(),
                         min_eigenvalue=np.linalg.eigvalsh(H).min()))

pd.DataFrame(rows)

,task,act,max_asymmetry,min_eigenvalue
0,digits01,sigmoid,3.552714e-15,1.000000
1,digits01,atan,7.105427e-15,1.000000
2,breast_cancer,sigmoid,7.105427e-15,1.005457
3,breast_cancer,atan,1.421085e-14,1.006126


In [5]:
#check 4 - fit the models
#scipy trust-ncg minimizes OUR objective using OUR gradient + HVP
#for sigmoid, sklearn newton-cg with C = 1/lam should land on the same w (sklearn_w_diff)
#w_norm for atan grows like lam^(-1/2), and atan's E is negative since its loss has no lower bound
rows = []
for name, (Xtr, Xte, ytr, yte, _) in data.items():
    for act in (SIGMOID, ATAN):
        for lam in LAMBDAS:
            f = BinaryObjective(Xtr, ytr, lam, act)

            r = minimize(f.objective, np.zeros(Xtr.shape[1]), method="trust-ncg",
                         jac = lambda w: f.gradient_and_curvature(w)[0],
                         hessp = lambda w, p: f.hessian_vector_product(p, f.gradient_and_curvature(w)[1]),
                         options={"gtol": 1e-9, "maxiter": 500})

            row = dict(task=name, act=act.name, lam=lam, E=r.fun,
                       grad_norm=np.linalg.norm(f.gradient_and_curvature(r.x)[0]), iters=r.nit,
                       w_norm=np.linalg.norm(r.x[:-1]), test_acc=accuracy(Xte, yte, r.x),
                       test_logloss=log_loss(Xte, yte, r.x, act))

            if act is SIGMOID:
                #sklearn's line search gives up at machine precision for small lam, w diff shows it still agrees
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    sk = LogisticRegression(C = 1/lam, solver="newton-cg", tol=1e-10, max_iter=10000).fit(Xtr[:, :-1], ytr)
                row["sklearn_w_diff"] = np.abs( np.r_[sk.coef_.ravel(), sk.intercept_] - r.x ).max()

            rows.append(row)

pd.DataFrame(rows).set_index(["task", "act", "lam"])

E     grad_norm  iters      w_norm  \
task          act     lam                                                  
digits01      sigmoid 0.01     0.097384  5.113500e-11     14    3.926955   
                      0.10     0.575050  2.153022e-11     13    2.934361   
                      1.00     3.026636  8.717445e-10     12    2.044986   
                      10.00   13.624880  1.151295e-07     11    1.291242   
              atan    0.01  -539.497804  2.454301e-09     17   92.705363   
                      0.10  -440.555209  5.650022e-09     13   29.314745   
                      1.00  -341.645783  3.505491e-10     14    9.266186   
                      10.00 -243.040252  8.142360e-10     11    2.919701   
breast_cancer sigmoid 0.01     8.731828  5.417324e-09     20   24.217244   
                      0.10    16.280959  9.178767e-10     16    8.559877   
                      1.00    27.059159  8.867548e-09     13    3.595525   
                      10.00   51.321286  3.517391e-07     12    1.787106   
              atan    0.01  -713.845708  6.953564e-09     97  107.825559   
                      0.10  -589.848714  2.853458e-10     24   32.368992   
                      1.00  -467.811378  1.411234e-09     14   10.458052   
                      10.00 -336.657632  3.258670e-07     13    3.419911   

                             test_acc  test_logloss  sklearn_w_diff  
task          act     lam                                            
digits01      sigmoid 0.01   1.000000      0.000285    1.498045e-08  
                      0.10   1.000000      0.001134    1.220281e-07  
                      1.00   1.000000      0.004575    2.765315e-10  
                      10.00  1.000000      0.020299    2.800392e-09  
              atan    0.01   1.000000      0.001077             NaN  
                      0.10   1.000000      0.003410             NaN  
                      1.00   1.000000      0.010829             NaN  
                      10.00  1.000000      0.034540             NaN  
breast_cancer sigmoid 0.01   0.951049      0.219173    1.184889e-06  
                      0.10   0.951049      0.103141    1.660818e-09  
                      1.00   0.958042      0.085874    2.079057e-09  
                      10.00  0.958042      0.113924    1.131892e-08  
              atan    0.01   0.944056      0.250845             NaN  
                      0.10   0.937063      0.193134             NaN  
                      1.00   0.958042      0.125654             NaN  
                      10.00  0.944056      0.138207             NaN